# Data Loading and Processing

This notebook demonstrates how to load and process raw hyperspectral data for use with the `spectral_select` library.

**What you'll learn:**
- Load raw `.im3` hyperspectral files
- Apply spectral cutoffs (Rayleigh and second-order)
- Normalize by exposure time and laser power
- Save processed data to pickle format
- Load processed data with `SpectraData`

## 1. Setup

Import the data processing utilities from the `scripts` module:

In [ ]:
import sys
from pathlib import Path

# Add scripts to path (if running from notebooks/examples)
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / "scripts"))

from data_processing import HyperspectralProcessor, HyperspectralDataLoader
from spectral_select import SpectraData

## 2. Configure Paths

Set up paths to your raw data and metadata files:

In [ ]:
# === UPDATE THESE PATHS FOR YOUR DATA ===

# Directory containing .im3 files (named by excitation wavelength, e.g., "365.0.im3")
DATA_DIR = project_root / "Data" / "Raw" / "Lichens_2"

# Excel file with exposure times (columns: Excitation, Exposure)
METADATA_PATH = DATA_DIR / "metadata.xlsx"

# Excel file with laser power measurements (columns: Excitation Wavelength (nm), Average Power (W))
LASER_POWER_PATH = DATA_DIR / "TLS Scans" / "average_power.xlsx"

# Output directory for processed data
OUTPUT_DIR = project_root / "Data" / "processed" / "LichensProcessed"

# Spectral cutoff offset in nm
CUTOFF_OFFSET = 60

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

## 3. Method A: Full Pipeline Processing

Use `HyperspectralProcessor` for a complete processing pipeline including:
- Spectral cutoff (Rayleigh + second-order)
- Exposure time normalization
- Laser power normalization

In [ ]:
# Create processor
processor = HyperspectralProcessor(
    data_path=str(DATA_DIR),
    metadata_path=str(METADATA_PATH),
    laser_power_excel=str(LASER_POWER_PATH),
    cutoff_offset=CUTOFF_OFFSET,
    verbose=True,
)

# Run full pipeline
output_files = processor.process_full_pipeline(
    output_dir=str(OUTPUT_DIR),
    exposure_reference="max",  # Normalize to max exposure time
    power_reference="min",     # Normalize to min laser power
    create_parquet=False,
    preserve_full_data=True,
)

print("\nGenerated files:")
for name, path in output_files.items():
    print(f"  {name}: {path}")

In [ ]:
# View data summary
processor.print_summary()

## 4. Method B: Step-by-Step Loading

For more control, use `HyperspectralDataLoader` to load and process data step by step:

In [ ]:
# Create loader (requires ImageJ/Fiji for .im3 files)
loader = HyperspectralDataLoader(
    data_path=str(DATA_DIR),
    metadata_path=str(METADATA_PATH),
    cutoff_offset=CUTOFF_OFFSET,
    use_fiji=True,
    verbose=True,
)

# Load data with spectral cutoff
data = loader.load_data(apply_cutoff=True, pattern="*.im3")

print(f"\nLoaded {len(loader.excitation_wavelengths)} excitation wavelengths:")
print(loader.excitation_wavelengths)

In [ ]:
# Access data for a specific excitation
excitation = loader.excitation_wavelengths[0]
cube, wavelengths = loader.get_cube(excitation, processed=True)

print(f"\nData for excitation {excitation}nm:")
print(f"  Cube shape: {cube.shape} (height, width, bands)")
print(f"  Emission wavelengths: {wavelengths[:5]}... ({len(wavelengths)} total)")

In [ ]:
# Visualize the cutoff effect
loader.visualize_cutoff(excitation)

In [ ]:
# Save to pickle
output_file = OUTPUT_DIR / "data_manual_processing.pkl"
loader.save_to_pkl(str(output_file))
print(f"Saved to: {output_file}")

## 5. Load Processed Data with SpectraData

Once data is saved to pickle, load it with `SpectraData.from_pickle()`:

In [ ]:
# Load the processed pickle file
# Use the file generated by the processor or loader
pkl_path = OUTPUT_DIR / f"data_cutoff_{CUTOFF_OFFSET}nm_exposure_max_power_min.pkl"

data = SpectraData.from_pickle(pkl_path)

print(f"Sample: {data.sample_name}")
print(f"Spatial shape: {data.spatial_shape}")
print(f"Number of excitations: {data.n_excitations}")
print(f"Excitation wavelengths: {data.excitation_wavelengths}")

In [ ]:
# Access per-excitation data
for ex_nm in data.excitation_wavelengths:
    ex_data = data.get_excitation(ex_nm)
    print(f"Ex {ex_nm:.0f}nm: {ex_data.n_bands} bands, shape {ex_data.shape}")

## 6. Visualize Spectra

Use the loader's visualization utilities to explore the data:

In [ ]:
# Plot mean emission spectrum for all excitations
loader.visualize_spectrum(excitation=None, processed=True)

In [ ]:
# Visualize a single band image
loader.visualize_image(excitation=365.0, emission_wavelength=500.0)

In [ ]:
# Create false color visualization
loader.visualize_false_color(excitation=365.0, method='pca')

## Expected File Structure

Your raw data directory should look like this:

```
Data/Raw/YourSample/
├── 310.0.im3        # Named by excitation wavelength
├── 325.0.im3
├── 340.0.im3
├── 365.0.im3
├── 385.0.im3
├── 400.0.im3
├── 415.0.im3
├── 430.0.im3
├── metadata.xlsx    # Columns: Excitation, Exposure
└── TLS Scans/
    └── average_power.xlsx  # Laser power measurements
```

## Pickle File Format

The saved pickle file contains:

```python
{
    'data': {
        '310.0': {
            'cube': np.ndarray,       # (H, W, bands)
            'wavelengths': [420.0, 430.0, ...],
            'excitation': 310.0,
            'raw': {...},
            'exposure_normalization_factor': float,
            'laser_power_normalization_factor': float,
        },
        '325.0': {...},
        ...
    },
    'excitation_wavelengths': [310.0, 325.0, ...],
    'metadata': {...},
    'cutoff_offset': 40,
}
```

## Summary

In this notebook, you learned how to:

1. **Full pipeline** with `HyperspectralProcessor.process_full_pipeline()`
2. **Step-by-step** with `HyperspectralDataLoader.load_data()`
3. **Save** processed data with `loader.save_to_pkl()`
4. **Load** with `SpectraData.from_pickle()`
5. **Visualize** spectra and images

**Next steps:**
- See `01_quickstart.ipynb` for running wavelength selection analysis
- See `02_validation.ipynb` for validating with ground truth